# Importando nuevos metodos
En esta sesion aprenderemos como importar un metodo nuevo que incluye factores de caracterizacion nuevos

In [1]:
from utils import custom_methods_importer
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from rich import print, pretty
from pathlib import Path
pretty.install()
# del bd.databases['biosphere3']

In [2]:
bd.projects.set_current("peru25")
bd.databases


Databases dictionary with 5 object(s):
        biosphere3
        db_bici
        ecoinvent39
        mi_base_de_datos
        new_biosphere3

In [12]:
# Si no tienes biosphere, puedes hacer 
# bi.bw2setup()

In [3]:
nombre_biosphere = 'new_biosphere3'

In [4]:
if nombre_biosphere in bd.databases:
    del bd.databases[nombre_biosphere]
# bd.Database('ecoinvent-3.9.1-biosphere').copy(nombre_biosphere)
bd.Database('biosphere3').copy(nombre_biosphere)

Vacuuming database 


100%|██████████| 4711/4711 [00:00<00:00, 6464.02it/s] 


Vacuuming database 


Brightway2 SQLiteBackend: new_biosphere3

In [5]:
custom_methods_importer(
    lcia_file='plastics_existing.xlsx', 
    biosphere_name='new_biosphere3',
    overwrite=True
)

 ✅ Node Nitrogen oxides already exists, passing 

 ⚠ Node `Poliestireno` does not exist, creating new one... 

 ⚠ Node `Polietileno de baja densidad y de baja densidad lineal` does not exist, creating new one... 

 ⚠ Node `Polietileno tereftalato` does not exist, creating new one... 

 ✅ Node Nitric oxide already exists, passing 

 ✅ Node Nitrogen oxides already exists, passing 

 ⚠ Node `Polietileno de alta densidad` does not exist, creating new one... 

 ✅ Node Ammonia already exists, passing 

 ⚠ Node `Polipropileno` does not exist, creating new one... 

Applying strategy: normalize_units
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applied 4 strategies in 0.18 seconds
1 methods
9 cfs
0 unlinked cfs
Wrote matching file to:
/home/jupyter-summer25peru_glarr-180f6/.local/share/Brightway3/peru25.01fb3473/output/lcia-matching-errors_custom_lcia.xlsx
Wrote 1 LCIA methods with 9 characterization factors


In [16]:
m=bd.Method(('plastics','microplastic category','microplastic_acum'))
m.load()


[
    (145998072969031680, 1.39),
    (145998072855785472, 1.0),
    (145998072537018368, 0.12),
    (145998072457326592, 1.21),
    (145998072654458880, 0.51),
    (145997979104702464, 39.0),
    (145997978957901871, 123.0),
    (145997978479751187, 42.0),
    (145997977993211959, 0.5)
]

In [17]:
m.metadata


{
    'description': '',
    'filename': 'plastics_existing.xlsx',
    'unit': 'MJ HDPEeq',
    'abbreviation': 'plasticsmm.a07b99fe69bb6a205a9ca39d9e4c85d8',
    'num_cfs': 9,
    'geocollections': ['world']
}

# Probando nuestro nuevo metodo

Creamos una nueva biblioteca de la construccion de la bicicleta

In [26]:
if 'db_bici' in bd.databases: # borramos alguna base de datos existente con el mismo nombre para estar seguros
    del bd.databases['db_bici']

db = bd.Database('db_bici') 
db.register()
db.make_searchable()
# bd.Database(nombre_biosphere).get('co2').delete()

In [27]:
data = {
    'code': 'bici',
    'name': 'produccion bici',
    'location': 'PE',
    'unit': 'piece'
}

bike = db.new_activity(**data)
bike.save()

data = {
    'code': 'CF',
    'name': 'carbon fibre',
    'unit': 'kilogram',
    'location': 'CN'
}

cf = db.new_activity(**data)
cf.save()

ng = db.new_activity(
    name="Nat Gas", 
    code='ng', 
    location='NO', 
    unit='MJ'
)

ng.save()

co2 = bd.Database(nombre_biosphere).new_activity(
    name="Carbon Dioxide", 
    code='co2', 
    categories=('air',),
    type='emission',
)

co2.save()
print('Estas son las actividades: ',list(db))

## Aqui creamos las aristas
bike.new_exchange(
    amount=2.5, 
    type='technosphere',
    input=cf
).save()

cf.new_exchange(
    amount=237.3, 
    type='technosphere',
    input=ng,
).save()

ng.new_exchange(
    amount=26.6 / 237, 
    type='biosphere',
    input=co2,
).save()
print('Estos las aristas de bike: ', list(bike.exchanges()))



Estas son las actividades: 
['produccion bici' (piece, PE, None), 'Nat Gas' (MJ, NO, None), 'carbon fibre' (kilogram, CN, None)]

Estos las aristas de bike: 
[Exchange: 2.5 kilogram 'carbon fibre' (kilogram, CN, None) to 'produccion bici' (piece, PE, None)>]

Ahora agregamos las nuevas aristas que se conectan a los nodos de biosphere relevantes en nuestro nuevo metodo

> 3 de Polietileno de alta densidad-('water',)
> 
> 2 de 9990b51b-7023-4700-bca0-1a32ef921f74


In [28]:
bike.new_exchange(
    amount=3, 
    type='biosphere',
    input=bd.Database('new_biosphere3').get("Polietileno de alta densidad-('water',)"),
).save()

bike.new_exchange(
    amount=2, 
    type='biosphere',
    input=bd.Database('new_biosphere3').get('9990b51b-7023-4700-bca0-1a32ef921f74'),
).save()
print('Esta es la nueva lista de exchanges de bike: ', list(bike.exchanges()))

Esta es la nueva lista de exchanges de bike: 
[
    Exchange: 2.5 kilogram 'carbon fibre' (kilogram, CN, None) to 'produccion bici' (piece, PE, None)>,
    Exchange: 3 kilogram 'Polietileno de alta densidad' (kilogram, GLO, ('water',)) to 'produccion bici' (piece, 
PE, None)>,
    Exchange: 2 kilogram 'Ammonia' (kilogram, None, ('air', 'urban air close to ground')) to 'produccion bici' 
(piece, PE, None)>
]

In [29]:
mi_nuevo_asombroso_metodo = ('plastics','microplastic category','microplastic_acum')

In [31]:
lca = bc.LCA({bike.id:1},method=mi_nuevo_asombroso_metodo) # Instancia la clase
lca.lci() # calcula el inventario de ciclo de vida
lca.lcia() # Calcula los impactos 
print("El impacto es: ", lca.score)

ValueError: LCA can only be performed on products, not activities (145999194723053568 is the wrong dimension)

# Otra categoria de impacto

In [91]:
custom_methods_importer(
    lcia_file='plastics_existing_and_biotic.xlsx', 
    biosphere_name='new_biosphere3',
    overwrite=True
)

 ✅ Node Ammonia already exists, passing 

 ✅ Node Anchoveta peruana already exists, passing 

 ✅ Node Polietileno de alta densidad already exists, passing 

 ✅ Node Nitrogen oxides already exists, passing 

 ✅ Node Argentines already exists, passing 

...

 ✅ Node Polietileno de baja densidad y de baja densidad lineal already exists, passing 

 ✅ Node Poliestireno already exists, passing 

 ✅ Node Nitrogen oxides already exists, passing 

 ✅ Node Polipropileno already exists, passing 

Applying strategy: normalize_units
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applied 4 strategies in 0.10 seconds
2 methods
14 cfs
0 unlinked cfs
Wrote matching file to:
/home/gustavo/.local/share/Brightway3/ex.54d54a12/output/lcia-matching-errors_custom_lcia.xlsx
Wrote 2 LCIA methods with 14 characterization factors


In [93]:
m_2=bd.Method(('Robinho world +','Biotic resource depletion', 'Depleted stock fraction potential'))
m_2.load()


[
    (('new_biosphere3', "Anchoveta peruana-('water', 'marine')"), 0.0),
    (('new_biosphere3', "Araucanian herring-('water', 'marine')"), 0.0),
    (('new_biosphere3', "Argentines-('water', 'marine')"), 0.0),
    (('new_biosphere3', "Armed snook-('water', 'marine')"), 0.0),
    (('new_biosphere3', "Atlantic pomfret-('water', 'marine')"), 2e-06)
]

In [94]:
ng.new_exchange(
    amount= 239940, 
    type='biosphere',
    input=bd.Database(nombre_biosphere).get("Anchoveta peruana-('water', 'marine')"),
).save()

ng.new_exchange(
    amount= 234740, 
    type='biosphere',
    input=bd.Database(nombre_biosphere).get("Atlantic pomfret-('water', 'marine')"),
).save()

In [95]:
mi_nuevo_aun_mas_asombroso_metodo = ('Robinho world +','Biotic resource depletion', 'Depleted stock fraction potential')

In [98]:
lca_2 = bc.LCA({bike:1},method=mi_nuevo_aun_mas_asombroso_metodo) # Instancia la clase
lca_2.lci() # calcula el inventario de ciclo de vida
lca_2.lcia() # Calcula los impactos 
print("El impacto es: ", lca_2.score)

El impacto es:  278.51901287865525